In [ ]:
:toolchain stable
:dep candle-notebooks = "0.1.0"
use candle_notebooks::*;
use std::fs;
let device = Device::Cpu;
// Normalize working directory for file I/O (images, datasets).
set_notebook_cwd().expect("failed to set notebook cwd");
fs::create_dir_all("images_store").ok();
println!("Toolchain: stable | Device: {:?} | CWD: {}", device, std::env::current_dir().unwrap().display());

# Simple Candle Tensor Operations and Image Display

This notebook demonstrates basic candle tensor operations and saving tensors as images.

In [ ]:
// Environment Setup
// Clean solution: single :dep by name via Cargo patch, no paths
:dep candle-notebooks = "0.1.0"

// Import required modules
use candle_notebooks::{Tensor, Device, DType};
use candle_notebooks as nb;

println!("✓ Dependencies loaded: candle-notebooks");

In [ ]:
// Standard notebook initialization with working directory and image store management
candle_notebooks::set_notebook_cwd().unwrap();
candle_notebooks::set_image_store_rel_dir("images_store").unwrap();
std::fs::create_dir_all("images_store").ok();

println!("✓ Notebook initialized successfully!");
println!("  Working directory: {:?}", std::env::current_dir().unwrap());
println!("  Image store: images_store/");
println!("  Location: research/notebooks/candle_notebooks/simple_tensors.ipynb");

In [ ]:
use candle_notebooks::{Tensor, Device, DType};
use anyhow::Result;
use std::path::Path;

// Simple save_image function (adapted for anyhow::Result)
pub fn save_image<P: AsRef<std::path::Path>>(img: &Tensor, p: P) -> Result<()> {
    let p = p.as_ref();
    let (channel, height, width) = img.dims3()?;
    if channel != 3 {
        return Err(anyhow::anyhow!("save_image expects an input of shape (3, height, width)"));
    }
    let img = img.permute((1, 2, 0))?.flatten_all()?;
    let pixels = img.to_vec1::<u8>()?;
    let image_buffer: image::ImageBuffer<image::Rgb<u8>, Vec<u8>> =
        match image::ImageBuffer::from_raw(width as u32, height as u32, pixels) {
            Some(image) => image,
            None => return Err(anyhow::anyhow!("error saving image {p:?}")),
        };
    image_buffer.save(p)?;
    Ok(())
}

println!("Image save function loaded!");

Image save function loaded!


In [5]:
// Ensure images_store directory exists
std::fs::create_dir_all("images_store")?;
println!("images_store directory ready");

images_store directory ready


In [6]:
// Create device
let device = Device::Cpu;
println!("Using device: {:?}", device);

Using device: Cpu


In [7]:
// Create a simple RGB gradient tensor
let height = 100;
let width = 100;

// Create red gradient (horizontal)
let mut red_values: Vec<f32> = Vec::with_capacity(height*width);
for _y in 0..height {
    for x in 0..width { red_values.push((x as f32 / width as f32) * 255.0); }
}
let red_channel = Tensor::from_vec(red_values, (height, width), &device)?;

// Create green gradient (vertical)
let mut green_values: Vec<f32> = Vec::with_capacity(height*width);
for y in 0..height {
    for _x in 0..width { green_values.push((y as f32 / height as f32) * 255.0); }
}
let green_channel = Tensor::from_vec(green_values, (height, width), &device)?;

// Blue channel - diagonal gradient
let mut blue_values: Vec<f32> = Vec::with_capacity(height*width);
for y in 0..height {
    for x in 0..width { blue_values.push(((x + y) as f32 / (width + height) as f32) * 255.0); }
}
let blue_channel = Tensor::from_vec(blue_values, (height, width), &device)?;

println!("Created RGB gradient channels: {}x{}", height, width);

Created RGB gradient channels: 100x100


In [13]:
// Stack channels into RGB image tensor (3, height, width)
let rgb_tensor = Tensor::stack(&[&red_channel, &green_channel, &blue_channel], 0)?;
let rgb_u8 = rgb_tensor.to_dtype(DType::U8)?;

println!("RGB tensor shape: {:?}", rgb_u8.shape());
println!("RGB tensor dtype: {:?}", rgb_u8.dtype());

RGB tensor shape: [3, 100, 100]
RGB tensor dtype: U8
RGB tensor dtype: U8


In [17]:
// Save the tensor as an image
let output_path = "images_store/gradient_test.png";
save_image(&rgb_u8, output_path)?;
println!("Saved RGB gradient to: {}", output_path);

Saved RGB gradient to: images_store/gradient_test.png


In [18]:
// Create a checkerboard pattern
let size = 64;
let checker_size = 8;

let mut checkerboard_data: Vec<f32> = Vec::with_capacity(size*size);
for y in 0..size {
    for x in 0..size {
        let checker_x = x / checker_size;
        let checker_y = y / checker_size;
        checkerboard_data.push(if (checker_x + checker_y) % 2 == 0 { 255.0 } else { 0.0 });
    }
}

let checkerboard = Tensor::from_vec(checkerboard_data, (size, size), &device)?;

// Make it RGB by repeating the pattern for all 3 channels
let checker_rgb = Tensor::stack(&[&checkerboard, &checkerboard, &checkerboard], 0)?;
let checker_u8 = checker_rgb.to_dtype(DType::U8)?;

save_image(&checker_u8, "images_store/checkerboard.png")?;
println!("Saved checkerboard pattern to: images_store/checkerboard.png");

Saved checkerboard pattern to: images_store/checkerboard.png


In [ ]:
// Create some random noise
let noise_size = 128;
let mut noise_data: Vec<f32> = Vec::with_capacity(noise_size * noise_size * 3);
for i in 0..noise_size*noise_size*3 {
    noise_data.push(((i * 7919) % 256) as f32);
}

let noise_tensor = Tensor::from_vec(noise_data, (3, noise_size, noise_size), &device)?;
let noise_u8 = noise_tensor.to_dtype(DType::U8)?;

save_image(&noise_u8, "images_store/noise.png")?;
println!("Saved noise pattern to: images_store/noise.png");

In [ ]:
// Basic tensor operations demonstration
let a = Tensor::new(&[[1f32, 2.], [3., 4.]], &device)?;
let b = Tensor::new(&[[5f32, 6.], [7., 8.]], &device)?;

let sum = (&a + &b)?;
let product = (&a * &b)?;
let matmul = a.matmul(&b)?;

println!("Matrix A: {:?}", a.to_vec2::<f32>()?);
println!("Matrix B: {:?}", b.to_vec2::<f32>()?);
println!("A + B: {:?}", sum.to_vec2::<f32>()?);
println!("A * B (element-wise): {:?}", product.to_vec2::<f32>()?);
println!("A @ B (matrix multiply): {:?}", matmul.to_vec2::<f32>()?);

## Summary

This notebook demonstrates:
1. Basic tensor creation and operations
2. Creating RGB image tensors with gradients and patterns
3. Saving tensors as image files using the `save_image` function
4. Working with different data types (F32 -> U8 conversion)

Images are saved under the `images_store/` directory:
- `images_store/gradient_test.png` - RGB gradient demonstration
- `images_store/checkerboard.png` - Black and white checkerboard pattern
- `images_store/noise.png` - Pseudo-random noise pattern